# Question 10 - 45. Jump Game II

You are given a **0-indexed** array of integers `nums` of length `n`. You are initially positioned at `nums[0]`.

Each element `nums[i]` represents the maximum length of a forward jump from index `i`. In other words, if you are at `nums[i]`, you can jump to any `nums[i + j]` where:

- `0 <= j <= nums[i]` and
- `i + j < n`

Return the **minimum number of jumps** to reach `nums[n - 1]`. The test cases are generated such that you can reach `nums[n - 1]`.

**Example 1:**

    Input: nums = [2,3,1,1,4]
    Output: 2
    Explanation: The minimum number of jumps to reach the last index is 2. Jump 1 step from index 0 to 1, then 3 steps to the last index.

**Example 2:**

    Input: nums = [2,3,0,1,4]
    Output: 2

**Constraints:**

- `1 <= nums.length <= 10^4`
- `0 <= nums[i] <= 1000`
- It's guaranteed that you can reach `nums[n - 1]`.

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Greedy 'rounds' (BFS by levels without a queue)

📘 **Revise first:** [Pattern C · running best-so-far](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (try every jump) | O(2ⁿ) | O(n) |
| 1½ DP (min jumps to each index) | O(n²) | O(n) |
| 2️⃣ Optimized (count rounds of reach) | O(n) | O(1) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

The same stepping stones as Question 9, but now you're **promised** you can reach the end. The question is: **what's the fewest number of jumps?**

```
nums = [2, 3, 1, 1, 4]
index 0 → index 1 (1 square) → index 4 (3 squares)   = 2 jumps
```

**Everyday picture: ripples, or "rounds" of reach.**
- **Round 0:** you're on stone 0.
- **Round 1:** every stone you can reach with **one** jump from stone 0. Because jumps can be shorter, that's a solid stretch.
- **Round 2:** every new stone you can reach with **one more** jump from *any* stone in round 1. Again a solid stretch, just further along.
- …keep going until a round includes the last stone. **The round number is the answer.**

You never need to decide *which* stone to jump from. You only need to know **where each round ends** and **how far the next round can stretch**.

### Step 0 · Clarify before coding

🗣️ *"Same setup as Jump Game, values are maximum jump lengths, but now reaching the end is guaranteed, and I need the minimum number of jumps."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Always reachable? | **Yes**, guaranteed. | No "impossible" case to handle. |
| Only one element? | Already at the end → **0** jumps. | |
| Shorter jumps allowed? | **Yes.** | Each round's reachable stones form one solid stretch. |
| Path or count? | Count only. | |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** from each stone, try every jump, recursively find the fewest jumps from where you land, and add 1. Take the minimum.

🗣️ *"Brute force recursively tries every jump from each index and takes one plus the minimum over the landing spots. It's exponential, because the same index gets solved again from many different paths."*

In [1]:
class SolutionBrute:
    def jump(self, nums: list[int]) -> int:
        last = len(nums) - 1

        def fewest_from(i: int) -> float:
            if i >= last:
                return 0
            best = float("inf")
            for step in range(1, nums[i] + 1):
                best = min(best, 1 + fewest_from(i + step))
            return best

        return fewest_from(0)

#### Step 1½ · Remember answers (dynamic programming)

`dp[j]` = fewest jumps to reach stone `j`. From each stone `i`, every stone it can reach gets `dp[i] + 1` if that's better.

🗣️ *"Caching gives one answer per index, but each index updates up to nums[i] neighbours, so it's O(n²) worst case. I think the 'rounds' idea gets this to linear."*

In [2]:
class SolutionDP:
    def jump(self, nums: list[int]) -> int:
        n = len(nums)
        dp = [float("inf")] * n
        dp[0] = 0
        for i in range(n):
            for j in range(i + 1, min(i + nums[i], n - 1) + 1):
                dp[j] = min(dp[j], dp[i] + 1)
        return dp[-1]

### Step 2 · Optimize: the "aha" moment

"Fewest steps" usually means **BFS** (breadth-first search: explore everything 1 step away, then everything 2 steps away, and so on). Normally BFS needs a queue. But here:

> 💡 **Aha:** each BFS "round" is a **solid stretch of indices**, so I can describe a whole round with **just its right edge**. No queue needed.

Walk `i` from left to right, keeping:
- `round_end`: the last stone reachable with the jumps counted so far.
- `farthest`: the furthest stone reachable with **one more** jump, from any stone seen so far.
- `jumps`: rounds used so far.

When `i` reaches `round_end`, I've looked at every stone in this round, so I **must** jump again: `jumps += 1`, and the new round ends at `farthest`.

**Why loop only up to the second-to-last stone?** Once you're standing on the last stone you don't jump again. If the last stone happened to equal `round_end`, we'd count one jump too many.

🗣️ *"Fewest jumps is a shortest-path question, so I think in BFS levels. Because jumps are maximums, each level is a contiguous range of indices, so I don't need a queue. I scan left to right, tracking the end of the current level and the farthest index the next level can reach. When I reach the end of the current level, I count a jump and the next level ends at farthest. Every index is visited once, so it's O(n) time and O(1) space."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| Try every jump (recursion) | ❌ | Exponential. |
| DP array `dp[i]` | ⚠️ | Correct, but O(n²) updates and O(n) memory. |
| Real BFS with a queue + visited set | ⚠️ | The right idea, but pushing every neighbour can be O(n²), plus O(n) memory. |
| Dijkstra / heap | ❌ | Overkill. Every jump costs exactly 1, so plain BFS is already optimal. |
| **Rounds as ranges (`round_end`, `farthest`)** | ✅ **best** | BFS compressed to two numbers. One pass, no containers. |

**Why greedy is correct:** every stone in the same round costs the **same** number of jumps. So which stone in the round you jump from doesn't change the count. Only how far the round can reach matters.

In [3]:
class Solution:
    def jump(self, nums: list[int]) -> int:
        jumps = 0
        round_end = 0      # last index reachable using `jumps` jumps
        farthest = 0       # farthest index reachable using one more jump
        for i in range(len(nums) - 1):          # never jump FROM the last index
            farthest = max(farthest, i + nums[i])
            if i == round_end:                  # finished scanning this round
                jumps += 1
                round_end = farthest
                if round_end >= len(nums) - 1:  # next round contains the goal
                    break
        return jumps

### Step 3 · Toy example walkthrough (tell it like a story)

`nums = [2, 3, 1, 1, 4]`, last index = 4

| i | nums[i] | farthest (one more jump) | i == round_end? | jumps | round_end |
|---|---|---|---|---|---|
| start | | 0 | | 0 | 0 |
| 0 | 2 | 2 | ✅ round 0 done → jump | **1** | 2 |
| 1 | 3 | **4** | ❌ | 1 | 2 |
| 2 | 1 | 4 | ✅ round 1 done → jump | **2** | 4 → covers the end, stop |

Answer **2** ✅

**The rounds, drawn:**
```
index:    0 | 1 2 | 3 4
round:    0 |  1  |  2      ← the last index is in round 2 → 2 jumps
```

**Narrated:** "From stone 0 I can reach stones 1 and 2, so that's round 1. While scanning round 1, stone 1 can reach all the way to stone 4. When I finish round 1, I take my second jump, and the next round reaches stone 4. Two jumps."

In [4]:
cases = [
    ([2, 3, 1, 1, 4], 2),
    ([2, 3, 0, 1, 4], 2),
    ([0], 0),                 # already there
    ([1, 2], 1),
    ([1, 1, 1, 1], 3),
    ([10, 1, 1, 1, 1], 1),
    ([1, 2, 1, 1, 1], 3),
]
for nums, expected in cases:
    for Impl in (SolutionBrute, SolutionDP, Solution):
        assert Impl().jump(nums) == expected, Impl.__name__
    print(nums, "->", expected)

import random
for _ in range(500):
    nums = [random.randint(1, 3) for _ in range(random.randint(1, 10))]   # always reachable
    assert Solution().jump(nums) == SolutionBrute().jump(nums), nums
print("All tests passed ✅")

[2, 3, 1, 1, 4] -> 2
[2, 3, 0, 1, 4] -> 2
[0] -> 0
[1, 2] -> 1
[1, 1, 1, 1] -> 3
[10, 1, 1, 1, 1] -> 1
[1, 2, 1, 1, 1] -> 3
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| Try every jump | exponential | O(n) call stack | Explodes. |
| DP array | O(n²) | O(n) | Every stone updates all the stones it can reach. |
| **Rounds** | **O(n)** | **O(1)** | Each stone is scanned once; three numbers are remembered. |

**Why O(n) time?** The loop visits each index once with constant work. `round_end` and `farthest` only ever move forward.

**Why O(1) memory?** Three integers, whatever the input size.

**Edge cases to mention:** one element (0 jumps), a huge first jump (1 jump), all ones (n − 1 jumps).

**Likely follow-ups:**
- *What if the end might be unreachable?* → If a round ends and `farthest` didn't move past it, return −1.
- *Return the path?* → Remember which index produced each round's `farthest`.
- *How does it connect to Question 9?* → Q9 asks whether the end is inside the reachable area at all. Q10 asks how many rounds it takes.

---

#### 🗣️ Full interview script (about 60 seconds)

*"Now the end is guaranteed reachable and I need the fewest jumps.*

*Brute force tries every jump recursively, which is exponential; a DP array of minimum jumps per index is O(n²).*

*Fewest jumps is really breadth-first search: everything reachable in one jump, then two, and so on. Because jump values are maximums, each of those levels is a contiguous range of indices, so I can represent a level by its right edge instead of using a queue. I scan left to right, tracking the current level's end and the farthest index reachable with one more jump. When I hit the current level's end, I count a jump and set the new end to farthest. I stop before the last index so I don't count a jump from the goal itself.*

*Each index is visited once: O(n) time, O(1) space."*